# 🌾 Krishi Mitra — Groq Edition with Rotating API Keys

**Fix applied:** Multiple Groq API keys rotate automatically:
- **Proactive switch** at 95,000 tokens — moves to the next key before hitting the 100k wall
- **Rate-limit handling** — on 429 errors, instantly tries the next key (no 60s wait)
- **Reactive fallback** — handles daily quota errors if they still occur

**How to get free Groq API keys:**
1. Go to https://console.groq.com
2. Sign up with a different email/Google account for each key
3. Each account gives you 100,000 tokens/day on llama-3.3-70b — so 3 keys = 300,000 tokens/day
4. Paste all your keys in Section 0

**Requirements:**
- Google Colab with T4 GPU
- 1–5 Groq API keys (more keys = more tokens per day)
- HuggingFace account + write token → https://hf.co/settings/tokens

---
> ⚠️ **Before running:** Enable GPU → Runtime > Change runtime type > T4 GPU

## ⚙️ SECTION 0 — Mount Drive & Enter API Keys

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_PATH = '/content/drive/MyDrive/KrishiMitra'
os.makedirs(DRIVE_PATH, exist_ok=True)
os.makedirs(f'{DRIVE_PATH}/dataset', exist_ok=True)
os.makedirs(f'{DRIVE_PATH}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_PATH}/final_model', exist_ok=True)
print('✅ Drive mounted')
print(f'   Saving all work to: {DRIVE_PATH}')

In [ ]:
from getpass import getpass

# ─────────────────────────────────────────────────────────────────────
# PASTE YOUR GROQ API KEYS HERE
# Get free keys at https://console.groq.com/keys
# Each account = 100,000 tokens/day on llama-3.3-70b (free, no card)
# Add as many keys as you have — 1 key minimum, 5 keys recommended
# ─────────────────────────────────────────────────────────────────────

print('Enter your Groq API keys one by one.')
print('Press Enter with blank input when done.\n')

GROQ_API_KEYS = []
i = 1
while True:
    key = getpass(f'Groq API key #{i} (or press Enter to finish): ').strip()
    if not key:
        break
    if key.startswith('gsk_'):
        GROQ_API_KEYS.append(key)
        print(f'  ✅ Key #{i} added')
        i += 1
    else:
        print(f'  ⚠️  Invalid key format — Groq keys start with gsk_')

if not GROQ_API_KEYS:
    raise ValueError('You must enter at least one Groq API key!')

HF_TOKEN    = getpass('\nHuggingFace token (write access): ').strip()
HF_USERNAME = input('HuggingFace username: ').strip()

os.environ['HF_TOKEN'] = HF_TOKEN

print(f'\n✅ {len(GROQ_API_KEYS)} Groq key(s) loaded')
print(f'   Daily token budget : ~{len(GROQ_API_KEYS) * 100000:,} tokens')
print(f'   HuggingFace user   : {HF_USERNAME}')

## 📦 SECTION 1 — Install Dependencies

In [ ]:
print('Installing packages...')

!pip install -q transformers==4.40.0
!pip install -q datasets==2.19.0
!pip install -q peft==0.10.0
!pip install -q accelerate==0.29.3
!pip install -q bitsandbytes==0.43.1
!pip install -q trl==0.8.6
!pip install -q sentencepiece
!pip install -q groq
!pip install -q requests beautifulsoup4
!pip install -q PyPDF2
!pip install -q pandas openpyxl
!pip install -q huggingface_hub
!pip install -q gradio

print('\n✅ All packages installed')

In [ ]:
import torch
print('=== System Check ===')
print(f'GPU available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU name      : {gpu.name}')
    print(f'VRAM total    : {gpu.total_memory / 1e9:.1f} GB')
    print('\n✅ GPU ready')
else:
    print('\n⚠️  No GPU — go to Runtime > Change runtime type > T4 GPU')

## 🔄 SECTION 2 — Rotating Groq Client

In [ ]:
from groq import Groq
import json
import time
import re

# ─────────────────────────────────────────────────────────────────────
# Model options (all free tier):
#   llama-3.3-70b-versatile → best quality,  100k tokens/day per key
#   llama-3.1-8b-instant    → fast + cheap,  500k tokens/day per key
#   mixtral-8x7b-32768      → good quality,  500k tokens/day per key
#   gemma2-9b-it            → good quality,  500k tokens/day per key
# ─────────────────────────────────────────────────────────────────────
GROQ_MODEL            = 'llama-3.3-70b-versatile'
GROQ_RATE_LIMIT_SLEEP = 2.5   # seconds between calls (safe for 30 req/min)
PROACTIVE_SWITCH_AT   = 95000 # switch to next key BEFORE hitting 100k limit


class RotatingGroqClient:
    """
    Wraps multiple Groq API keys and rotates between them automatically.
    - PROACTIVE rotation: switches to the next key when usage >= 95,000
      tokens so you never actually hit the 100k wall.
    - REACTIVE rotation: on rate-limit (429), immediately try the next key.
      On daily quota hit, permanently mark the key and rotate.
    Tracks per-key usage so you always know how many tokens each key spent.
    """

    def __init__(self, api_keys, model=GROQ_MODEL, switch_threshold=PROACTIVE_SWITCH_AT):
        if not api_keys:
            raise ValueError('Provide at least one Groq API key')
        self.clients            = [Groq(api_key=k) for k in api_keys]
        self.keys               = api_keys
        self.model              = model
        self.current            = 0
        self.switch_threshold   = switch_threshold
        self.exhausted          = set()   # all currently-blocked keys
        self._permanently_done  = set()   # keys past threshold or daily quota (never retry)
        self.tokens_used        = [0] * len(api_keys)
        self.requests_made      = [0] * len(api_keys)

    # ── internal helpers ─────────────────────────────────────────────

    def _next_key(self):
        """Rotate to the next non-exhausted key."""
        for _ in range(len(self.clients)):
            self.current = (self.current + 1) % len(self.clients)
            if self.current not in self.exhausted:
                print(f'   🔄 Rotated to key #{self.current + 1}')
                return True
        return False   # all keys exhausted

    def _mark_exhausted(self, idx, permanent=True):
        self.exhausted.add(idx)
        if permanent:
            self._permanently_done.add(idx)
        label = 'permanently' if permanent else 'temporarily'
        print(f'   ⚠️  Key #{idx + 1} {label} blocked — '
              f'{len(self.exhausted)}/{len(self.clients)} keys blocked')

    # ── public method ────────────────────────────────────────────────

    def create(self, messages, max_tokens=3000, temperature=0.8):
        """
        Send a chat completion request, rotating keys when:
        1. A key's cumulative usage crosses the switch_threshold (95k default)
        2. A rate-limit (429) or quota error is returned by Groq
        On rate limit: immediately rotate to the next key.
        If ALL keys are rate-limited in one pass: wait 60s and retry.
        Raises RuntimeError only if all keys are permanently exhausted.
        """
        max_rounds = 3   # full rounds through all keys before giving up

        for round_num in range(max_rounds):
            keys_rate_limited_this_round = 0

            for _ in range(len(self.clients)):
                # Skip to a non-exhausted key
                if self.current in self.exhausted:
                    if not self._next_key():
                        # All keys blocked — check if it's permanent or just rate limits
                        if len(self._permanently_done) >= len(self.clients):
                            raise RuntimeError(
                                '🚫 All Groq API keys have hit their daily limit.\n'
                                '   Options:\n'
                                '   1. Wait until midnight UTC for limits to reset\n'
                                '   2. Add more Groq keys in Section 0\n'
                                '   3. Switch GROQ_MODEL to llama-3.1-8b-instant (500k/day)'
                            )
                        break  # all keys temporarily blocked, go to round wait

                try:
                    response = self.clients[self.current].chat.completions.create(
                        model=self.model,
                        messages=messages,
                        max_tokens=max_tokens,
                        temperature=temperature,
                    )
                    # Track usage
                    used = response.usage.total_tokens if response.usage else max_tokens
                    self.tokens_used[self.current]   += used
                    self.requests_made[self.current] += 1

                    # ── Proactive rotation: switch BEFORE hitting the 100k limit ──
                    if self.tokens_used[self.current] >= self.switch_threshold:
                        print(f'   ⚡ Key #{self.current+1} reached '
                              f'{self.tokens_used[self.current]:,} tokens '
                              f'(threshold: {self.switch_threshold:,}) — switching proactively')
                        self._mark_exhausted(self.current, permanent=True)
                        if not self._next_key():
                            print('   ⚠️  All keys past threshold — will keep using last key until hard error')

                    return response

                except Exception as e:
                    err = str(e).lower()

                    if 'rate_limit' in err or '429' in err:
                        # Rate-limited — temporarily block and try NEXT key immediately
                        keys_rate_limited_this_round += 1
                        print(f'   ⏳ Rate limit on key #{self.current+1} — trying next key...')
                        self._mark_exhausted(self.current, permanent=False)
                        if not self._next_key():
                            break  # all keys blocked, fall through to round wait
                        time.sleep(1)  # brief pause before trying next key
                        continue

                    elif 'quota' in err or 'exceeded' in err or '402' in err or 'tokens per day' in err:
                        # Daily quota hit — mark key permanently and rotate
                        self._mark_exhausted(self.current, permanent=True)
                        if not self._next_key():
                            raise RuntimeError(
                                '🚫 All Groq API keys exhausted for today. '
                                'Add more keys or wait until midnight UTC.'
                            )
                        continue

                    else:
                        # Unknown error — re-raise
                        raise e

            # All keys were rate-limited this round — wait and retry
            if keys_rate_limited_this_round > 0 and round_num < max_rounds - 1:
                wait = 60 * (round_num + 1)
                print(f'   ⏳ All keys rate-limited — waiting {wait}s before retry round {round_num+2}...')
                # Reset exhausted to only permanently-done keys (un-block rate-limited ones)
                self.exhausted = set(self._permanently_done)
                time.sleep(wait)
            elif len(self._permanently_done) >= len(self.clients):
                raise RuntimeError(
                    '🚫 All Groq API keys exhausted for today. '
                    'Add more keys or wait until midnight UTC.'
                )

        raise RuntimeError(
            '🚫 All retry rounds failed. All keys are rate-limited.\n'
            '   Options:\n'
            '   1. Wait a few minutes and re-run this cell\n'
            '   2. Add more Groq keys in Section 0\n'
            '   3. Switch GROQ_MODEL to llama-3.1-8b-instant (500k/day)'
        )

    # ── reporting ────────────────────────────────────────────────────

    def status(self):
        print('\n=== Groq API Key Usage ===')
        total_tokens   = 0
        total_requests = 0
        for i, (t, r) in enumerate(zip(self.tokens_used, self.requests_made)):
            if i in self._permanently_done:
                status = '❌ EXHAUSTED'
            elif i in self.exhausted:
                status = '⏳ Rate-limited'
            else:
                status = '✅ Active'
            pct = min(100, int(t / self.switch_threshold * 100)) if self.switch_threshold else 0
            key_preview = self.keys[i][:12] + '...'
            print(f'  Key #{i+1} ({key_preview}) | {status} | {t:,} tokens ({pct}%) | {r} requests')
            total_tokens   += t
            total_requests += r
        print(f'  ─────────────────────────────────────────')
        print(f'  Total : {total_tokens:,} tokens across {total_requests} requests')
        remaining_keys = len(self.clients) - len(self._permanently_done)
        print(f'  Keys remaining     : {remaining_keys}/{len(self.clients)}')
        print(f'  Switch threshold   : {self.switch_threshold:,} tokens per key')


# Initialize the rotating client
groq_client = RotatingGroqClient(GROQ_API_KEYS, model=GROQ_MODEL)

# Test all keys
print(f'Testing {len(GROQ_API_KEYS)} key(s)...\n')
for i, key in enumerate(GROQ_API_KEYS):
    try:
        test_client = Groq(api_key=key)
        res = test_client.chat.completions.create(
            model=GROQ_MODEL,
            messages=[{'role': 'user', 'content': 'Reply: OK'}],
            max_tokens=5
        )
        print(f'  Key #{i+1} : ✅ Working — {key[:12]}...')
    except Exception as e:
        print(f'  Key #{i+1} : ❌ Failed  — {key[:12]}... ({e})')

print(f'\n✅ Rotating client ready')
print(f'   Model              : {GROQ_MODEL}')
print(f'   Keys loaded        : {len(GROQ_API_KEYS)}')
print(f'   Token budget       : ~{len(GROQ_API_KEYS) * 100000:,} tokens/day')
print(f'   Switch threshold   : {PROACTIVE_SWITCH_AT:,} tokens per key')

In [ ]:
def generate_qa_groq(topic, context_text='', n=12, difficulty_mix=True):
    """
    Generate Q&A pairs using the rotating Groq client.
    Automatically switches API keys when limits are hit.
    """
    difficulty_note = ''
    if difficulty_mix:
        half = n // 2
        difficulty_note = f"""
Generate exactly:
- {half} SIMPLE questions (difficulty: "simple"): practical, what a farmer would ask
- {half} EXPERT questions (difficulty: "expert"): technical, what an agronomist would ask"""

    context_note = f'\n\nReference text:\n{context_text[:1500]}' if context_text else ''

    system_msg = (
        'You are an expert agriculture dataset creator for Indian farming. '
        'You ALWAYS respond with valid JSON only — no explanation, no markdown fences, no preamble.'
    )

    user_msg = f"""Generate exactly {n} Q&A pairs about: {topic}{context_note}
{difficulty_note}

Requirements:
- Indian farming context (Chhattisgarh, MP, central India)
- Answers at least 3 sentences, practical and actionable
- Include organic and chemical options where relevant
- Mention Indian government schemes when applicable

Respond with ONLY this JSON array, nothing else:
[
  {{
    "question": "...",
    "answer": "...",
    "context": "one sentence background",
    "difficulty": "simple" or "expert",
    "topic": "{topic[:30].lower().replace(' ', '_')}"
  }}
]"""

    for attempt in range(3):
        try:
            response = groq_client.create(
                messages=[
                    {'role': 'system', 'content': system_msg},
                    {'role': 'user',   'content': user_msg}
                ],
                max_tokens=3000,
                temperature=0.8,
            )

            raw = response.choices[0].message.content.strip()
            raw = re.sub(r'^```(json)?\s*', '', raw)
            raw = re.sub(r'\s*```$', '', raw).strip()
            start = raw.find('[')
            end   = raw.rfind(']') + 1
            if start == -1 or end == 0:
                raise ValueError('No JSON array found')

            pairs = json.loads(raw[start:end])
            for p in pairs:
                p['source'] = 'groq_generated'
                p.setdefault('context', '')
                p.setdefault('difficulty', 'simple')
                p.setdefault('topic', topic[:30])
            return pairs

        except RuntimeError as e:
            # All keys exhausted — stop and surface the error
            raise e

        except Exception as e:
            wait = 2 ** attempt
            print(f'    Attempt {attempt+1} failed: {e} — retrying in {wait}s')
            time.sleep(wait)

    return []

# Quick test
print('Testing Q&A generation with rotating client...')
test = generate_qa_groq('Paddy blast disease', n=2, difficulty_mix=False)
if test:
    print(f'✅ Generation working!')
    print(f'   Sample: {test[0]["question"]}')
    groq_client.status()
else:
    print('❌ Generation failed — check your keys')

## 🌱 SECTION 3 — Collect HuggingFace Datasets

In [ ]:
from datasets import load_dataset, Dataset
import pandas as pd

hf_samples = []

# ── AgriKnowBot ──────────────────────────────────────────────────────
print('[1/3] Loading AgriKnowBot...')
try:
    ds1 = load_dataset('AgriKnowBot/agriculture-qa-dataset', split='train')
    for row in ds1:
        q = str(row.get('question', row.get('input', ''))).strip()
        a = str(row.get('answer',   row.get('output', ''))).strip()
        if len(q) > 15 and len(a) > 40:
            hf_samples.append({'question': q, 'answer': a, 'context': '',
                                'topic': 'general_agriculture', 'difficulty': 'simple',
                                'source': 'agriknowbot_hf'})
    print(f'   ✅ {len(ds1):,} samples')
except Exception as e:
    print(f'   ⚠️  Skipped: {e}')

# ── Databricks Dolly ─────────────────────────────────────────────────
print('[2/3] Loading Dolly 15k (agriculture filter)...')
try:
    ds2 = load_dataset('databricks/databricks-dolly-15k', split='train')
    agri_kw = ['crop', 'farm', 'soil', 'plant', 'pest', 'fertiliz', 'harvest',
                'seed', 'irrigat', 'wheat', 'rice', 'paddy', 'disease', 'insect',
                'agricultur', 'vegetable', 'fruit', 'sow', 'cultivation']
    count = 0
    for row in ds2:
        instr = str(row.get('instruction', '')).lower()
        resp  = str(row.get('response', ''))
        if any(kw in instr for kw in agri_kw) and len(resp) > 60:
            hf_samples.append({'question': row['instruction'].strip(), 'answer': resp.strip(),
                                'context': str(row.get('context', '')).strip(),
                                'topic': 'general_agriculture', 'difficulty': 'simple',
                                'source': 'dolly_hf'})
            count += 1
    print(f'   ✅ {count:,} agriculture samples extracted')
except Exception as e:
    print(f'   ⚠️  Skipped: {e}')

# ── Crop recommendation ───────────────────────────────────────────────
print('[3/3] Loading crop recommendation...')
try:
    ds3 = load_dataset('soumitri/crop-recommendation', split='train')
    for row in ds3:
        crop = row.get('label', row.get('crop', ''))
        if crop:
            q = f'What soil and climate conditions are needed to grow {crop}?'
            a = (f'{crop} grows best with nitrogen {row.get("N","N/A")} kg/ha, '
                 f'phosphorus {row.get("P","N/A")} kg/ha, potassium {row.get("K","N/A")} kg/ha. '
                 f'Ideal temperature {row.get("temperature","N/A")}°C, '
                 f'humidity {row.get("humidity","N/A")}%, pH {row.get("ph","N/A")}.')
            hf_samples.append({'question': q, 'answer': a, 'context': f'Crop: {crop}',
                                'topic': 'crop_recommendation', 'difficulty': 'simple',
                                'source': 'crop_recommendation_hf'})
    print(f'   ✅ {len(ds3):,} samples')
except Exception as e:
    print(f'   ⚠️  Skipped: {e}')

print(f'\n📊 Total from HuggingFace: {len(hf_samples):,} samples')

## 🤖 SECTION 4 — Generate Q&A with Rotating Groq Keys

In [ ]:
generation_topics = [
    # ── Plant diseases ──────────────────────────────────────────────
    ('Paddy blast disease symptoms causes fungicide treatment central India', 16),
    ('Rice brown plant hopper BPH identification chemical management',        12),
    ('Rice sheath blight Rhizoctonia management schedule',                    12),
    ('Soybean yellow mosaic virus SYMV detection and control',                12),
    ('Soybean rust Phakopsora early detection and management',                10),
    ('Wheat yellow rust brown rust black rust identification India',          12),
    ('Wheat loose smut karnal bunt prevention seed treatment',                10),
    ('Cotton bollworm pink bollworm integrated pest management',              12),
    ('Cotton whitefly sucking pest chemical biological management',           10),
    ('Tomato leaf curl virus TLCV whitefly vector management',               10),
    ('Tomato early blight Alternaria late blight Phytophthora treatment',    10),
    ('Tomato bacterial wilt Ralstonia soil borne disease control',            8),
    ('Chilli anthracnose Colletotrichum fruit rot management',               10),
    ('Maize downy mildew fall armyworm Spodoptera India',                    10),
    ('Pigeonpea Arhar wilt sterility mosaic disease management',             10),
    ('Groundnut tikka late leaf spot stem rot Sclerotium',                   10),
    ('Onion purple blotch Alternaria thrips management',                      8),
    ('Mango anthracnose hoppers integrated management',                       8),

    # ── Crop management ─────────────────────────────────────────────
    ('Paddy rice cultivation best practices Chhattisgarh MP',               16),
    ('Kharif Rabi Zaid sowing calendar central India',                       12),
    ('Soil health pH correction black cotton soil Vertisol',                 12),
    ('Vermicompost organic fertilizer preparation farm level India',         10),
    ('Drip irrigation setup water scheduling maintenance',                   10),
    ('Integrated nutrient management INM rice wheat system',                 10),
    ('Weed management paddy herbicides manual methods',                      10),
    ('Seed treatment fungicide insecticide before sowing',                   10),
    ('Intercropping crop rotation benefits methods India',                   10),
    ('Micronutrient deficiency zinc boron iron correction',                  10),
    ('Rainwater harvesting farm pond construction MP CG',                     8),
    ('Organic farming PGS NOP certification benefits India',                  8),

    # ── Government schemes ──────────────────────────────────────────
    ('PM-KISAN eligibility eKYC registration installment 6000',             14),
    ('MSP crops procurement APMC mandi selling process',                    12),
    ('PMFBY Pradhan Mantri Fasal Bima Yojana crop insurance',               12),
    ('Soil health card SHC scheme get and use recommendations',             10),
    ('Kisan Credit Card KCC application limit interest',                    10),
    ('PM Krishi Sinchai Yojana irrigation subsidy drip sprinkler',           8),
    ('eNAM electronic national agriculture market online trading',           8),

    # ── Post harvest ────────────────────────────────────────────────
    ('Post harvest storage paddy wheat pulses hermetic bags',               10),
    ('Vegetable grading packing cold chain small farmers',                   8),
    ('FPO farmer producer organisation benefits registration',               8),
]

total_expected   = sum(n for _, n in generation_topics)
tokens_per_topic = 2000   # conservative estimate
total_tokens     = len(generation_topics) * tokens_per_topic
keys_needed      = max(1, total_tokens // 100000)

print(f'Topics           : {len(generation_topics)}')
print(f'Expected pairs   : ~{total_expected:,}')
print(f'Estimated tokens : ~{total_tokens:,}')
print(f'Keys available   : {len(GROQ_API_KEYS)}')
print(f'Keys needed      : ~{keys_needed}')

if len(GROQ_API_KEYS) >= keys_needed:
    print(f'\n✅ You have enough keys to complete in one run!')
else:
    print(f'\n⚠️  You may hit limits — consider adding {keys_needed - len(GROQ_API_KEYS)} more key(s)')
    print(f'   Or the notebook will pause and tell you when to resume.')

In [ ]:
# ── Checkpoint system — saves progress after every topic ─────────────
CHECKPOINT_FILE = f'{DRIVE_PATH}/dataset/generation_checkpoint.json'

def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, 'r') as f:
            data = json.load(f)
        print(f'📂 Checkpoint loaded: {len(data["completed"])} topics done, '
              f'{len(data["samples"]):,} samples so far')
        return data['completed'], data['samples']
    print('📂 No checkpoint found — starting fresh')
    return [], []

def save_checkpoint(completed, samples):
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump({'completed': completed, 'samples': samples}, f)

completed_topics, generated_samples = load_checkpoint()

# ── Main generation loop ──────────────────────────────────────────────
failed_topics = []
print('\nStarting Groq Q&A generation with key rotation...\n')

for i, (topic, n) in enumerate(generation_topics):

    if topic in completed_topics:
        print(f'[{i+1:02d}/{len(generation_topics)}] ⏭️  Already done: {topic[:50]}')
        continue

    print(f'[{i+1:02d}/{len(generation_topics)}] {topic[:58]}...')

    try:
        pairs = generate_qa_groq(topic, n=n, difficulty_mix=True)

        if pairs:
            generated_samples.extend(pairs)
            completed_topics.append(topic)
            save_checkpoint(completed_topics, generated_samples)
            print(f'       ✅ +{len(pairs)} pairs | total: {len(generated_samples):,}')
        else:
            failed_topics.append(topic)
            print(f'       ❌ No pairs returned')

    except RuntimeError as e:
        # All keys exhausted
        print(f'\n{e}')
        print(f'\nProgress saved at: {CHECKPOINT_FILE}')
        print(f'Completed {len(completed_topics)}/{len(generation_topics)} topics')
        print('Re-run this cell tomorrow (limits reset at midnight UTC) to continue.')
        break

    time.sleep(GROQ_RATE_LIMIT_SLEEP)

print(f'\n📊 Generation session complete!')
print(f'   Generated : {len(generated_samples):,} Q&A pairs')
print(f'   Topics    : {len(completed_topics)}/{len(generation_topics)} done')
if failed_topics:
    print(f'   Failed    : {len(failed_topics)}')
groq_client.status()

## 🌐 SECTION 5 — Scrape Government Portals

In [ ]:
import requests
from bs4 import BeautifulSoup

def scrape_page(url, timeout=12):
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
        res  = requests.get(url, headers=headers, timeout=timeout)
        res.raise_for_status()
        soup = BeautifulSoup(res.text, 'html.parser')
        for tag in soup(['nav', 'footer', 'script', 'style', 'header']):
            tag.decompose()
        paras = soup.find_all('p')
        text  = ' '.join(p.get_text(strip=True) for p in paras if len(p.get_text(strip=True)) > 50)
        return text[:2500]
    except Exception:
        return ''

portal_sources = [
    ('https://farmer.gov.in',       'Indian government farmer portal crop advisory'),
    ('https://agricoop.nic.in',     'Ministry of Agriculture India crop schemes'),
    ('https://www.niphm.gov.in',    'Plant Health Management pest advisory India'),
    ('https://pib.gov.in/newsite/erelcontent.aspx?relid=110654', 'PM-KISAN farmer benefits'),
]

scraped_samples = []
print('Scraping government portals...\n')

for url, topic in portal_sources:
    print(f'Scraping: {topic}...')
    text  = scrape_page(url)
    pairs = generate_qa_groq(topic, context_text=text if len(text) > 300 else '',
                             n=8, difficulty_mix=False)
    if pairs:
        scraped_samples.extend(pairs)
        print(f'  ✅ +{len(pairs)} pairs')
    time.sleep(GROQ_RATE_LIMIT_SLEEP)

print(f'\n📊 Portal Q&A: {len(scraped_samples):,}')

## 🦠 SECTION 6 — Disease Q&A from PlantVillage

In [ ]:
india_diseases = [
    ('Tomato', 'Early blight'),        ('Tomato', 'Late blight'),
    ('Tomato', 'Leaf mold'),           ('Tomato', 'Septoria leaf spot'),
    ('Tomato', 'Spider mites'),        ('Tomato', 'Target spot'),
    ('Tomato', 'Yellow leaf curl virus'), ('Tomato', 'Mosaic virus'),
    ('Tomato', 'Bacterial spot'),
    ('Corn',   'Cercospora gray leaf spot'), ('Corn', 'Common rust'),
    ('Corn',   'Northern leaf blight'),
    ('Potato', 'Early blight'),        ('Potato', 'Late blight'),
    ('Pepper bell', 'Bacterial spot'), ('Grape', 'Black rot'),
    ('Grape',  'Leaf blight'),         ('Apple', 'Apple scab'),
    ('Orange', 'Citrus greening'),     ('Peach', 'Bacterial spot'),
]

disease_samples = []
print(f'Generating disease Q&A for {len(india_diseases)} classes...\n')

for i, (crop, disease) in enumerate(india_diseases):
    print(f'[{i+1:02d}/{len(india_diseases)}] {crop} — {disease}')
    prompt = (f'5 Q&A pairs about {disease} in {crop} for Indian farmers. '
              f'Cover: symptoms, spread, organic remedy, chemical dosage, prevention. '
              f'Return ONLY JSON: [{{"question":"...","answer":"...",'
              f'"context":"...","difficulty":"simple","topic":"plant_disease"}}]')
    try:
        res = groq_client.create(
            messages=[
                {'role': 'system', 'content': 'Return ONLY valid JSON arrays. No markdown, no explanation.'},
                {'role': 'user',   'content': prompt}
            ],
            max_tokens=1500,
            temperature=0.7,
        )
        raw = res.choices[0].message.content.strip()
        raw = re.sub(r'^```(json)?\s*', '', raw)
        raw = re.sub(r'\s*```$', '', raw).strip()
        pairs = json.loads(raw[raw.find('['):raw.rfind(']')+1])
        for p in pairs:
            p['source'] = 'plantvillage_groq'
        disease_samples.extend(pairs)
        print(f'       ✅ +{len(pairs)} pairs')
    except RuntimeError as e:
        print(f'\n{e}')
        break
    except Exception as e:
        print(f'       ⚠️  Failed: {e}')
    time.sleep(GROQ_RATE_LIMIT_SLEEP)

print(f'\n📊 Disease Q&A: {len(disease_samples):,}')
groq_client.status()

## 🧹 SECTION 7 — Combine, Clean & Push to HuggingFace

In [ ]:
import pandas as pd

all_raw = hf_samples + generated_samples + scraped_samples + disease_samples
print(f'HuggingFace  : {len(hf_samples):,}')
print(f'Groq gen     : {len(generated_samples):,}')
print(f'Scraped      : {len(scraped_samples):,}')
print(f'Disease      : {len(disease_samples):,}')
print(f'─────────────────────')
print(f'Total raw    : {len(all_raw):,}')

In [ ]:
df = pd.DataFrame(all_raw)
for col in ['question','answer','context','topic','difficulty','source']:
    if col not in df.columns:
        df[col] = ''

df = df.fillna('')
df['question']   = df['question'].astype(str).str.strip()
df['answer']     = df['answer'].astype(str).str.strip()
df['context']    = df['context'].astype(str).str.strip()
df['difficulty'] = df['difficulty'].astype(str).str.strip()

before = len(df)
df = df[df['question'].str.len() > 15]
df = df[df['answer'].str.len() > 50]
df = df[df['answer'].str.len() < 3000]
df = df[df['question'] != df['answer']]
df = df.drop_duplicates(subset=['question'], keep='first')
df = df.reset_index(drop=True)

print(f'Before : {before:,}  →  After : {len(df):,}  (removed {before-len(df):,})')
print(f'\nBy source:')
print(df['source'].value_counts().to_string())
print(f'\nBy difficulty:')
print(df['difficulty'].value_counts().to_string())

In [ ]:
from huggingface_hub import login
from datasets import Dataset

df.to_csv(f'{DRIVE_PATH}/dataset/krishi_mitra_dataset.csv', index=False)
print(f'✅ CSV saved to Drive')

login(token=HF_TOKEN)
hf_dataset   = Dataset.from_pandas(df)
dataset_repo = f'{HF_USERNAME}/krishi-mitra-agriculture-qa'
hf_dataset.push_to_hub(dataset_repo, private=False)

print(f'\n✅ Dataset live: https://huggingface.co/datasets/{dataset_repo}')
print(f'   Total: {len(hf_dataset):,} samples')

## 🔥 SECTION 8 — Format for LLaMA Fine-Tuning

In [ ]:
def format_for_llama(row):
    context = f'\nContext: {row["context"]}' if str(row.get('context','')).strip() else ''
    if str(row.get('difficulty','simple')) == 'expert':
        system = ('You are Krishi Mitra, an expert agronomist specializing in Indian agriculture. '
                  'Provide technically accurate, evidence-based recommendations.')
    else:
        system = ('You are Krishi Mitra (कृषि मित्र), a helpful farming assistant for Indian farmers. '
                  'Give simple, practical advice. Suggest organic and chemical options. '
                  'Mention government schemes where applicable.')
    return {
        'text': (f'### System:\n{system}\n\n'
                 f'### Instruction:{context}\n\n'
                 f'### Question:\n{row["question"]}\n\n'
                 f'### Answer:\n{row["answer"]}')
    }

formatted_ds = Dataset.from_pandas(df.apply(format_for_llama, axis=1, result_type='expand'))
split        = formatted_ds.train_test_split(test_size=0.05, seed=42)
train_ds     = split['train']
val_ds       = split['test']

print(f'Train : {len(train_ds):,}  |  Val : {len(val_ds):,}')
print(f'\nSample:\n{train_ds[0]["text"][:500]}')

## 🦙 SECTION 9 — Load LLaMA-2-7B in 4-bit

In [ ]:

import os, torch, gc
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

gc.collect()
torch.cuda.empty_cache()

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    low_cpu_mem_usage=True,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True
)
model.config.use_cache = False

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'right'

print("✅ Mistral loaded successfully!")

## 🔧 SECTION 10 — Apply QLoRA

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=32, lora_alpha=64,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout=0.05, bias='none', task_type='CAUSAL_LM'
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print(f'VRAM after LoRA: {torch.cuda.memory_allocated()/1e9:.1f} GB')

## 🚀 SECTION 11 — Train

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir=f'{DRIVE_PATH}/checkpoints',
    num_train_epochs=3,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    fp16=True,
    optim='paged_adamw_8bit',
    logging_dir=f'{DRIVE_PATH}/logs',
    logging_steps=20,
    report_to='none',
    save_strategy='steps',
    save_steps=100,
    save_total_limit=3,
    evaluation_strategy='steps',
    eval_steps=100,
    load_best_model_at_end=True,
    group_by_length=True,
    dataloader_num_workers=2,
)

trainer = SFTTrainer(
    model=model, train_dataset=train_ds, eval_dataset=val_ds,
    args=training_args, dataset_text_field='text',
    max_seq_length=512, peft_config=lora_config,
)

print(f'Train: {len(train_ds):,} | Val: {len(val_ds):,}')
print(f'Est. time: 2–4 hours on T4\n')
print('Starting...')

result = trainer.train()
print(f'\n✅ Done! Loss: {result.training_loss:.4f} | Time: {result.metrics["train_runtime"]/60:.1f} min')

## 💾 SECTION 12 — Save Model

In [ ]:
adapter_path = f'{DRIVE_PATH}/final_model/krishi-mitra-adapter'
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f'✅ Saved to Drive: {adapter_path}')

model_repo = f'{HF_USERNAME}/krishi-mitra-llama2-7b-qlora'
model.push_to_hub(model_repo)
tokenizer.push_to_hub(model_repo)
print(f'✅ Pushed to: https://huggingface.co/{model_repo}')

## 🧪 SECTION 13 — Test the Model

In [ ]:
def ask_krishi_mitra(question, context='', difficulty='simple', max_new_tokens=350):
    system = ('You are Krishi Mitra, an expert agronomist.' if difficulty == 'expert'
              else 'You are Krishi Mitra (कृषि मित्र), a helpful farming assistant.')
    ctx    = f'\nContext: {context}' if context else ''
    prompt = (f'### System:\n{system}\n\n### Instruction:{ctx}\n\n'
              f'### Question:\n{question}\n\n### Answer:\n')
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             temperature=0.7, do_sample=True, top_p=0.9,
                             repetition_penalty=1.1, pad_token_id=tokenizer.eos_token_id)
    full = tokenizer.decode(out[0], skip_special_tokens=True)
    idx  = full.rfind('### Answer:')
    return full[idx + len('### Answer:'):].strip() if idx != -1 else full.strip()

tests = [
    ('My paddy has brown spots on leaves, what disease?', 'Chhattisgarh, August', 'simple'),
    ('Recommended fungicide for blast in Kharif rice?',  'irrigated paddy',      'expert'),
    ('How to register for PM-KISAN?',                    '',                     'simple'),
    ('Tomato leaves curling with white flies underneath','MP, summer season',    'simple'),
]

print('=' * 65)
for q, ctx, diff in tests:
    print(f'\n[{diff.upper()}] {q}')
    if ctx: print(f'Context: {ctx}')
    print(f'Answer : {ask_krishi_mitra(q, ctx, diff)[:350]}')
    print('-' * 65)

## 📱 SECTION 14 — Gradio Chatbot

In [ ]:
import gradio as gr

def respond(message, history, location, crop, user_type):
    ctx = ''
    if location.strip(): ctx += f'Location: {location.strip()}. '
    if crop and crop != 'Select crop': ctx += f'Crop: {crop}.'
    return ask_krishi_mitra(message, context=ctx,
                            difficulty='expert' if user_type == 'Agronomist / Expert' else 'simple')

with gr.Blocks(title='Krishi Mitra') as demo:
    gr.Markdown('# 🌾 Krishi Mitra — कृषि मित्र\nPlant diseases · Crop advice · Govt schemes')
    with gr.Row():
        with gr.Column(scale=1):
            location  = gr.Textbox(label='Location', placeholder='e.g. Raipur, CG')
            crop      = gr.Dropdown(
                            ['Select crop','Paddy','Wheat','Soybean','Maize',
                             'Cotton','Tomato','Chilli','Potato','Onion','Other'],
                            value='Select crop', label='Crop')
            user_type = gr.Radio(['Farmer','Agronomist / Expert'], value='Farmer', label='I am a')
        with gr.Column(scale=2):
            gr.ChatInterface(
                fn=respond, additional_inputs=[location, crop, user_type],
                examples=[
                    ['My paddy has brown spots — what disease?'],
                    ['Mere gehu mein pila rust hai kya karoon?'],
                    ['How to register for PM-KISAN?'],
                    ['Best fertilizer dose for soybean kharif?'],
                ], title=''
            )

demo.launch(share=True)
print('\n✅ Chatbot launched!')

## 🔁 SECTION 15 — Resume After Disconnect

In [ ]:
# Run only if session disconnected mid-training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch, glob

DRIVE_PATH = '/content/drive/MyDrive/KrishiMitra'
MODEL_ID   = 'NousResearch/Llama-2-7b-hf'
checkpoints = sorted(glob.glob(f'{DRIVE_PATH}/checkpoints/checkpoint-*'))

if checkpoints:
    latest = checkpoints[-1]
    bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True,
                                     bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16)
    model     = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map='auto')
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    tokenizer.pad_token = tokenizer.eos_token
    model = PeftModel.from_pretrained(model, latest)
    print(f'✅ Resumed from: {latest}')
    print('Now run: trainer.train(resume_from_checkpoint=latest)')
else:
    print('No checkpoints found — run from Section 9.')

---
## ✅ Summary

| Component | Detail |
|---|---|
| API key rotation | **Proactive** switch at 95k tokens + instant rotation on rate-limit |
| Checkpoint system | Saves after every topic — resume anytime |
| Token tracking | Per-key usage + % shown after each session |
| Keys supported | 1–10 keys (each adds ~95k usable tokens/day) |
| Model | LLaMA-2-7B QLoRA fine-tuned |
| Dataset | 5k–10k agriculture Q&A pairs |

**If you hit limits again:**
1. Add more Groq keys in Section 0
2. Switch `GROQ_MODEL` to `llama-3.1-8b-instant` (500k tokens/day per key)
3. Re-run Section 4 — checkpoint will skip completed topics automatically